# R-011: published-reference reconciliation

Assignment `9-r1-g1-r011-reference-20261003-a6f29c8d`. Read only accepted v0.5.8 evidence at `3ca66b4c119d8968c1ede7769125ef8b3a44fc61`; scientific execution commit `feaabea5d4408d14f15aee47da5399f5d2130be8`.

This analytical notebook reads, extracts, joins, counts and reports. It never imports COBRA or runs an optimization, author simulation, original notebook or benchmark. Source files downloaded to the excluded `.r011-source-cache` remain external; source registry records URLs/hashes. For a fresh extraction, retrieve the exact registry resources into that cache and use a new output destination; the writer refuses existing generated outputs. Published graphs are manual visual evidence and are not a complete numerical reference.


In [1]:
from pathlib import Path
import collections, hashlib, json, sys, xml.etree.ElementTree as ET
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT / 'scripts'))
import r011_audit_tools_v1 as audit
cache = ROOT / '.r011-source-cache'
registry = audit.load(ROOT, audit.OUT + 'source_registry.json')
ds = audit.dataset(ROOT)
gene_names = audit.model_gene_names(ROOT)
print('Read-only analytical environment:', sys.executable, sys.version.split()[0])
print('Dataset 2 records:', len(ds), 'joint-gene records:', sum(len(d['genes']) > 1 for d in ds.values()), 'conditional records:', sum(d['conditional'] for d in ds.values()))
print('Source layers:', ', '.join(sorted({s['kind'] for s in registry['sources']})))


Read-only analytical environment: C:\Users\pc\anaconda3\envs\auxoyeast-repro\python.exe 3.12.14
Dataset 2 records: 147 joint-gene records: 3 conditional records: 14
Source layers: accepted_project_execution, author_code_model_snapshot, experimental_dataset, official_supplements, primary_article, published_aggregate, systematic_positive_prediction_graph


## Published coverage and source identity

The full paper and all three official supplements were inspected. Table 1 provides selected pre-curation error classes; sections 3.2 provide two explicit corrected pairs. Fig. 2 supplies aggregates, and Figs. 4/S2 supply systematic positive-prediction relationships. Dataset 1 and Dataset 2 supply experimental phenotypes, not author-model classifications. No complete versioned benchmark prediction matrix was found in this inspected package. This is a bounded search result, not proof such a matrix does not exist elsewhere. Fig. S2 was inspected as a graph; exhaustive edge digitization was not performed.


In [2]:
import openpyxl
source_profiles = []
for name in ['mmc2.xlsx', 'mmc3.xlsx']:
    book = openpyxl.load_workbook(cache / name, read_only=True, data_only=True)
    for sheet in book:
        values = list(sheet.iter_rows(values_only=True))
        source_profiles.append(dict(file=name, sheet=sheet.title, max_row=sheet.max_row, max_column=sheet.max_column, nonempty_data_rows=sum(any(v is not None for v in r) for r in values[1:]), headers=list(values[0])))
    book.close()
assert (cache / 'mmc3.xlsx').read_bytes() == (ROOT / 'data/raw/mmc3.xlsx').read_bytes()
audit.write_json(ROOT, audit.OUT + 'source_profiles.json', source_profiles)
print(json.dumps(source_profiles, indent=2))
print('Official Dataset 2 downloaded bytes equal the fixed project input.')


[
  {
    "file": "mmc2.xlsx",
    "sheet": "Sheet1",
    "max_row": 681,
    "max_column": 9,
    "nonempty_data_rows": 657,
    "headers": [
      "Gene",
      "Gene Systematic Name",
      "Experiment Type",
      "Experiment Type Category",
      "Mutant Information",
      "Strain Background",
      "Chemical",
      "Details",
      "Reference"
    ]
  },
  {
    "file": "mmc3.xlsx",
    "sheet": "all",
    "max_row": 148,
    "max_column": 6,
    "nonempty_data_rows": 147,
    "headers": [
      "Gene Systematic Name",
      "Chemical",
      "exchange",
      "ID",
      "Strain Background",
      "Reference"
    ]
  },
  {
    "file": "mmc3.xlsx",
    "sheet": "amino acid",
    "max_row": 74,
    "max_column": 4,
    "nonempty_data_rows": 73,
    "headers": [
      "Gene Systematic Name",
      "Chemical",
      "exchange",
      "ID"
    ]
  },
  {
    "file": "mmc3.xlsx",
    "sheet": "vitamin",
    "max_row": 29,
    "max_column": 4,
    "nonempty_data_rows": 28,
    "head

## Pair reference extraction

Table 1 rowspans are expanded programmatically; full compound sets are explicit. Gene aliases are resolved using the frozen SBML geneProduct systematic IDs and common-name labels, cross-checked against Dataset 1. The grouped ALD2 and ALD3 phrase remains ambiguous between individual and joint knockouts. References omitting a required background supplement remain non-comparable. Pair IDs are retained only to identify current rows; joins use model, full gene set, full rescue set and background gates.


In [3]:
article = (cache / 'article.xml').read_bytes()
assert audit.sha(article) == 'f7741f07375700e98508af8375f2d74cf9c8bd83da5ae38a33186140503a958b'
compound_ids = {'aspartate':['r_1881'], 'cysteine':['r_1883'], 'methionine':['r_1902'], 'methionine and threonine':['r_1902','r_1911'], 'choline':['r_1683'], 'd-Glucosamine':['r_4506'], 'glutathione':['r_1807'], 'ergosterol':['r_1757'], 'heme':['a_0001'], 'pantothenic acid':['r_1548'], 'thiamine':['r_2067'], 'uracil':['r_2090']}
references = []
for number, item in enumerate(audit.table1_entries(article), 1):
    names = item['raw_gene_field'].split(' and ')
    genes = sorted(g for name in names for g in gene_names[name])
    assert all(len(gene_names[name]) == 1 for name in names)
    ambiguous = item['raw_gene_field'] == 'ALD2 and ALD3'
    references.append(dict(reference_id='T1-%02d'%number, model='Yeast9', genes=genes, candidate_gene_sets=([gene_names[n] for n in names]+[genes]) if ambiguous else [genes], rescue_ids=compound_ids[item['raw_compound']], background_ids=None, identity_ambiguous=ambiguous, scope='benchmark', kind='selected_published_error_class', raw_label=item['raw_label'], raw_gene_label=item['raw_gene_field'], raw_compound=item['raw_compound'], label_semantics='paper_auxotrophy_classes', source_id='ART', url='https://pmc.ncbi.nlm.nih.gov/articles/PMC11704421/#tbl1', locator='Table 1 (tbl1), body row %d, %s-%s'%(item['table_row'],item['raw_gene_field'],item['raw_compound'])))
for name, compound, paragraph in [('AAT2','aspartate','p0160'),('MET13','methionine','p0175')]:
    references.append(dict(reference_id='TEXT-'+name, model='Yeast9_curated', genes=gene_names[name], rescue_ids=compound_ids[compound], background_ids=None, scope='benchmark', kind='explicit_corrected_pair_text', raw_label='correct', label_semantics='paper_auxotrophy_classes', source_id='ART', url='https://pmc.ncbi.nlm.nih.gov/articles/PMC11704421/', locator='Section 3.2, JATS paragraph '+paragraph+'; '+name+'-'+compound))
print('Selected benchmark labels:', len(references), '(24 Table 1 entries and 2 explicit curated-text entries)')


Selected benchmark labels: 26 (24 Table 1 entries and 2 explicit curated-text entries)


## Systematic graph scope

The following 41 Fig. 4 edges have clear visible endpoints. They are retained as positive systematic predictions, never interpreted as benchmark correctness or full-run class labels. The central crossed MET/TRR1 block and Fig. S2 are not exhaustively transcribed; omitted edges are explicitly unextracted, not inferred absent. A complete graph still lacks benchmark row conditions and negatives.


In [4]:
# Manual visual transcription of Fig. 4 clear branches; not numerical author outputs.
fig4 = [(['LYS2','LYS4','LYS5','LYS12','LYS1','LYS9'], 'lysine', 'r_1900'), (['TYR1'],'tyrosine','r_1913'), (['HIS7','HIS4','HIS1','HIS2','HIS6','HIS5','HIS3'],'histidine','r_1893'), (['ARG5,6','ARG4','ARG3','ARG7','ARG1','ARG8','ORT1'],'arginine','r_1879'), (['AAT2'],'aspartate','r_1881'), (['AAT2'],'asparagine','r_1880'), (['BAT1'],'valine','r_1914'), (['PHA2'],'phenylalanine','r_1903'), (['GLN1'],'glutamine','r_1891'), (['MET13','MET2'],'methionine','r_1902'), (['LEU2','LEU1','BAT2'],'leucine','r_1899'), (['THR4','THR1'],'threonine','r_1911'), (['TRP1','TRP4','TRP2','TRP5','TRP3'],'tryptophan','r_1912'), (['PRO3'],'proline','r_1904'), (['CYS3','CYS4'],'cysteine','r_1883')]
for names, compound, reaction in fig4:
    for name in names:
        assert len(gene_names[name]) == 1, name
        references.append(dict(reference_id='FIG4-'+name+'-'+compound, model='Yeast9_curated', genes=gene_names[name], rescue_ids=[reaction], background_ids=None, scope='systematic', kind='systematic_positive_edge', raw_label='positive auxotrophy edge', label_semantics='systematic_positive_prediction', source_id='FIG4', url='https://pmc.ncbi.nlm.nih.gov/articles/PMC11704421/#fig4', locator='Figure 4, visible '+name+' -> L-'+compound+' edge', extraction='manual visual; background and exact benchmark run not supplied'))
assert len(references) == 67
audit.write_json(ROOT, audit.OUT + 'reference_records.json', references)
print('Retained reference facts:', len(references), '; 41 graph edges kept distinct from 26 benchmark qualitative labels.')


Retained reference facts: 67 ; 41 graph edges kept distinct from 26 benchmark qualitative labels.


## Complete reconciliation and denominators

Unknown values are never counted as agreement or disagreement. `comparable_qualitative` means a published benchmark label can be compared qualitatively at gene/rescue/background level; it does not certify the exact author execution or precise numerical growth. `not_reported` means no benchmark-specific label found by this extraction in the inspected package. `not_comparable`, `ambiguous`, and `unavailable` remain separate. Supplemental unextracted graph edges are tracked as a coverage limitation.


In [5]:
records = audit.reconcile(ROOT, references)
assert len(records) == 294 and len({(r['model'],r['pair_id']) for r in records}) == 294
audit.write_csv(ROOT, audit.OUT + 'pair_crosswalk.csv', records, list(records[0]))
summary = audit.summarize(records)
audit.write_csv(ROOT, audit.OUT + 'coverage_summary.csv', summary, list(summary[0]))
mapping = audit.reference_mapping(ROOT, references)
audit.write_json(ROOT, audit.OUT + 'unmatched_and_multiple.json', mapping)
unmatched = mapping['unmatched_reference_records']
multiple = mapping['multiple_or_ambiguous_current_rows']
print(json.dumps(summary,indent=2))
print('Unmatched reference facts:',len(unmatched),'multiple/ambiguous current rows:',len(multiple))
print('Qualitative mismatches:',[(r['model'],r['pair_id'],r['current_class'],r['reference_class']) for r in records if r['agreement']=='mismatch'])


[
  {
    "model": "Yeast9",
    "total": 147,
    "comparable_qualitative": 21,
    "matches": 20,
    "mismatches": 1,
    "unknown": 126,
    "not_reported": 123,
    "unavailable": 0,
    "ambiguous": 1,
    "not_comparable": 2,
    "exact_run_comparable": 0
  },
  {
    "model": "Yeast9_curated",
    "total": 147,
    "comparable_qualitative": 2,
    "matches": 2,
    "mismatches": 0,
    "unknown": 145,
    "not_reported": 116,
    "unavailable": 0,
    "ambiguous": 0,
    "not_comparable": 29,
    "exact_run_comparable": 0
  }
]
Unmatched reference facts: 10 multiple/ambiguous current rows: 1
Qualitative mismatches: [('Yeast9', 'excel:95', 'type_I', 'type_II')]


## Aggregate comparison and bounded inference

Figure 2B gives Yeast9 107 KO non-growing pairs, 93 rescued and 14 not rescued; 40 KO-growing pairs. Curated: 119, 117, 2 and 28. Text section 3.3 gives 93/147 and 117/147, and rounded 63.27%/79.59%. Figure 2A original category labels sum to 91 (13+12+45+21), inconsistent with text/Fig. 2B; this is recorded without silently selecting or correcting numbers.

A one-unit correct-count difference does not identify one discordant pair. Conditional on aligned records and three-class meanings, the Fig. 2B class histograms imply a minimum of four differing original-model labels, and one curated-model label. This arithmetic lower bound identifies neither the pairs nor the cause.


In [6]:
aggregates=[]
for model, paper, category_counts in [('Yeast9',dict(correct=93,type_I=40,type_II=14),[13,12,45,21]), ('Yeast9_curated',dict(correct=117,type_I=28,type_II=2),[24,13,57,23])]:
    current = collections.Counter(r['current_class'] for r in records if r['model']==model)
    assert sum(paper.values())==147 and sum(current.values())==147
    lower=sum(abs(current[k]-paper[k]) for k in paper)//2
    aggregates.append(dict(model=model,denominator=147,current_correct=current['correct'],paper_correct_text_fig2b=paper['correct'],current_type_I=current['type_I'],paper_type_I_fig2b=paper['type_I'],current_type_II=current['type_II'],paper_type_II_fig2b=paper['type_II'],conditional_minimum_class_disagreements=lower,fig2a_category_sum=sum(category_counts),fig2a_internal_agreement=sum(category_counts)==paper['correct'],source_id='FIG2',source_locator='Section 3.3 p0185/p0195; Fig. 2B branches; Fig. 2A category labels',inference='Histogram lower bound only, assuming same 147 records and class definitions; does not identify pairs/causes'))
audit.write_csv(ROOT, audit.OUT + 'aggregate_comparison.csv',aggregates,list(aggregates[0]))
print(json.dumps(aggregates,indent=2))


[
  {
    "model": "Yeast9",
    "denominator": 147,
    "current_correct": 92,
    "paper_correct_text_fig2b": 93,
    "current_type_I": 37,
    "paper_type_I_fig2b": 40,
    "current_type_II": 18,
    "paper_type_II_fig2b": 14,
    "conditional_minimum_class_disagreements": 4,
    "fig2a_category_sum": 91,
    "fig2a_internal_agreement": false,
    "source_id": "FIG2",
    "source_locator": "Section 3.3 p0185/p0195; Fig. 2B branches; Fig. 2A category labels",
    "inference": "Histogram lower bound only, assuming same 147 records and class definitions; does not identify pairs/causes"
  },
  {
    "model": "Yeast9_curated",
    "denominator": 147,
    "current_correct": 116,
    "paper_correct_text_fig2b": 117,
    "current_type_I": 28,
    "paper_type_I_fig2b": 28,
    "current_type_II": 3,
    "paper_type_II_fig2b": 2,
    "conditional_minimum_class_disagreements": 1,
    "fig2a_category_sum": 117,
    "fig2a_internal_agreement": true,
    "source_id": "FIG2",
    "source_locator": 

## Protocol and static model context

Read source code, XML and JSON as data only. Regressions excel:58/59 (CPA1/CPA2, arginine plus uracil) are distinct from Table 1 URA2-uracil. Their relationship to a changed GPR is context, not evidence that they explain the paper discrepancy. Objective and glucose state below are static model facts, not newly computed WT rates.


In [7]:
protocol = audit.load(ROOT,audit.OUT+'protocol_evidence.json')
audit.write_csv(ROOT,audit.OUT+'protocol_comparison.csv',protocol,list(protocol[0]))
static=[]
ns='{http://www.sbml.org/sbml/level3/version1/core}'
fbc='{http://www.sbml.org/sbml/level3/version1/fbc/version2}'
for model, filename in [('Yeast9','yeast9.0.xml'),('Yeast9_curated','Yeast9_curated.xml')]:
    tree=ET.fromstring(audit.read(ROOT,'data/raw/'+filename))
    parameters={e.attrib['id']:e.attrib.get('value') for e in tree.iter(ns+'parameter')}
    reactions={e.attrib['id'].removeprefix('R_'):e for e in tree.iter(ns+'reaction')}
    item=dict(model=model,glucose_lower_bound=parameters[reactions['r_1714'].attrib[fbc+'lowerFluxBound']],model_sha256=audit.sha(audit.read(ROOT,'data/raw/'+filename)),r0250_association_xml=ET.tostring(reactions['r_0250'].find(fbc+'geneProductAssociation'),encoding='unicode'))
    template=audit.load(ROOT,'results/model_cache/'+model+'.from_sbml.json') if (ROOT/('results/model_cache/'+model+'.from_sbml.json')).exists() else None
    item['objectives']=[e.attrib for e in tree.iter(fbc+'fluxObjective')]
    static.append(item)
audit.write_json(ROOT,audit.OUT+'static_model_context.json',static)
focus=[r for r in records if r['pair_id'] in ['excel:58','excel:59','excel:95','excel:146','excel:147']]
audit.write_csv(ROOT,audit.OUT+'focus_pairs.csv',focus,list(focus[0]))
print('Protocol topics:',len(protocol));print(json.dumps(static,indent=2))
print('Focus:',[(r['model'],r['pair_id'],r['genes'],r['requested_background'],r['comparability'],r['agreement']) for r in focus])


Protocol topics: 12
[
  {
    "model": "Yeast9",
    "glucose_lower_bound": "-1",
    "model_sha256": "0e120b0d4015048ef2edaf86ea039c533a72827ff00a34ca35d9fbe87a2781e5",
    "r0250_association_xml": "<ns0:geneProductAssociation xmlns:ns0=\"http://www.sbml.org/sbml/level3/version1/fbc/version2\">\n          <ns0:and>\n            <ns0:geneProductRef ns0:geneProduct=\"YJL130C\" />\n            <ns0:geneProductRef ns0:geneProduct=\"YJR109C\" />\n            <ns0:geneProductRef ns0:geneProduct=\"YOR303W\" />\n          </ns0:and>\n        </ns0:geneProductAssociation>\n      ",
    "objectives": [
      {
        "{http://www.sbml.org/sbml/level3/version1/fbc/version2}reaction": "R_r_2111",
        "{http://www.sbml.org/sbml/level3/version1/fbc/version2}coefficient": "1"
      }
    ]
  },
  {
    "model": "Yeast9_curated",
    "glucose_lower_bound": "-1",
    "model_sha256": "d0bd57f6ec99b8fb1f0ffeb7006ba88a879a9185100236249d6e4bae333a82af",
    "r0250_association_xml": "<ns0:geneProductA

## Matching checks and execution label

The edge cases test missing labels, duplicate identity, joint genes, different rescue/background, conflicting labels and non-benchmark positive graphs. Validation checks the actual preserved input bytes and exact 294-key recomputation. The recorded execution is direct Python execution of these new analytical cells with the reference interpreter, **not a Jupyter-kernel run**. No claim is made for unrun checks.


In [8]:
tests = audit.self_tests()
audit.write_json(ROOT,audit.OUT+'join_validation.json',dict(passed=all(tests.values()),checks=tests))
print(json.dumps(tests,indent=2))
print('PASS:',sum(tests.values()),'join edge-case tests; benchmark/optimization/author simulation NOT_RUN.')


{
  "missing_reference": true,
  "full_multigene_condition": true,
  "single_gene_not_joint_knockout": true,
  "different_background": true,
  "unspecified_background": true,
  "different_rescue": true,
  "ambiguous_duplicate_mapping": true,
  "conflicting_labels": true,
  "different_label_semantics": true,
  "inaccessible_reference": true,
  "systematic_graph_not_benchmark": true
}
PASS: 11 join edge-case tests; benchmark/optimization/author simulation NOT_RUN.


## روند استدلال / reasoning record

Evidence: full article, official datasets and supplementary graphs, dated upstream tree/code/models, and accepted project CSV/sidecars. Decisions: require full gene/rescue identity, preserve background and model scope, compare only explicit qualitative benchmark classes, and count unknowns separately. Rejected shortcuts: using experiment as model prediction, gene-only or row-order joins, interpreting a positive graph as a full benchmark matrix, treating 92 vs 93 as exactly one changed pair, or assigning CPA1/CPA2 regressions as the cause. Limits: unpublished part_temp.xlsx and numerical author predictions, unpinned run/version/conditions, graph extraction coverage, Fig. 2A aggregate inconsistency. Neither complete scientific equivalence nor a causal explanation for the correct-count gap is established. A bounded next step would request the authors' versioned 147-row prediction/growth table and matching part_temp.xlsx; do not run it in this assignment.
